<a href="https://colab.research.google.com/github/hansjio/tsektxt-model-training/blob/master/notebooks/colab_present.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# TsekTxt: defense presentation notebook
Read-only and fast. It never trains a model and never writes to GitHub or Hugging Face. Runtime > Change runtime type > T4 GPU is faster, but CPU also works. Run the cells top to bottom (a few minutes in total; most of that is downloading the three models once).

## 1. What the thesis tests, and what this notebook shows
**Question.** Can three fine-tuned language models (RoBERTa-Tagalog, mBERT, XLM-RoBERTa) tell apart news articles labelled *credible* from those labelled *not credible*, and how do they behave on short Taglish election posts?

**Labels.** `1 = credible`, `0 = not credible`. *Not credible* is the **positive class** for precision, recall and F1.

**Two kinds of data.**
- *Internal news set*: long news articles, split 70 / 15 / 15 into train / validation / test.
- *Taglish election posts*: real short posts collected separately. This set is **test only**: no model is trained, tuned or validated on it.

**Important reading rules.** Labels follow the source (fact-checkers and news outlets), so a score says how well a model reproduces those labels, not that it "detects misinformation". Each model was fine-tuned once (single seed), and RoBERTa-Tagalog vs XLM-RoBERTa is **not** a significant difference (McNemar p = 0.55), so the notebook does not rank them.

**This notebook shows:** the fine-tuning setup (3), training behaviour (4), internal test results (5), the limits of those results (6), Taglish results if ready (7), live predictions (8), Integrated Gradients (9), and a summary (10). Every number is read from a file in `results/`.

## 2. Setup
Clones the code and results from GitHub (skipped if the folder is already there, so re-running never nests folders) and installs two libraries at the versions the demo app was checked with. Torch, pandas, scikit-learn and matplotlib already come with Colab. Takes about a minute.

You may see *"Could not find cuda drivers"* or other TensorFlow lines: that is TensorFlow noise from Colab, not an error, and this notebook does not use TensorFlow.

In [ ]:
import os
os.environ["USE_TF"] = "0"                 # do not load TensorFlow
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
if os.path.basename(os.getcwd()) != "tsektxt-model-training":
    if not os.path.isdir("tsektxt-model-training"):
        !git clone -q https://github.com/hansjio/tsektxt-model-training.git
    %cd tsektxt-model-training
!pip -q install "transformers==4.57.6" "captum==0.9.0" pyyaml
import torch
print("Device:", "GPU (" + torch.cuda.get_device_name(0) + ")" if torch.cuda.is_available() else "CPU")

## 3. Fine-tuning setup
All three models start from a pre-trained checkpoint and are fine-tuned on the training split with **one fixed configuration** taken from the methodology chapter (`configs/config.yaml`). There was **no hyperparameter search**: the same settings were used for every model so that differences come from the models, not from per-model tweaking. Validation loss is used only for early stopping and to keep the best epoch.

In [ ]:
import yaml, json, pandas as pd
from pathlib import Path
from IPython.display import display, Markdown, Image
R = Path("results")
cfg = yaml.safe_load(open("configs/config.yaml"))
t = cfg["training"]
rows = [
    ("Random seed", cfg["seed"], "Single run per model"),
    ("Split (train / val / test)", "{train} / {val} / {test}".format(**cfg["data"]["split_ratio"]), "Test set is never used for training or early stopping"),
    ("Max length (tokens)", t["max_length"], "Longer text is cut"),
    ("Learning rate", t["learning_rate"], "AdamW with linear warm-up"),
    ("Warm-up ratio", t["warmup_ratio"], "First 10% of steps ramp the learning rate up"),
    ("Weight decay", t["weight_decay"], "Mild regularisation"),
    ("Train batch size", t["train_batch_size"], ""),
    ("Max epochs", t["max_epochs"], "Upper limit; early stopping usually ends sooner"),
    ("Early stopping patience", t["early_stopping_patience"], "Stop after this many epochs without lower validation loss"),
    ("Class-weighted loss", t["class_weighted_loss"], "Counteracts the class imbalance"),
    ("Positive class", "not_credible (0)", "Used for precision / recall / F1"),
]
display(pd.DataFrame(rows, columns=["Setting", "Value", "Why / note"]))
display(pd.DataFrame([(k, v) for k, v in cfg["models"].items()], columns=["Model key", "Pre-trained checkpoint"]))

## 4. Training behaviour
Top: loss on the training data and on the validation data after each epoch. When validation loss stops improving while training loss keeps falling, the model starts to memorise; early stopping ends training and the best epoch is kept. The table lists, per model, the epoch with the lowest validation loss. Training logs come from `results/<model>/training_log.json`.

In [ ]:
if (R / "figures" / "loss_curves.png").exists():
    display(Image(filename=str(R / "figures" / "loss_curves.png")))
NAMES = ["roberta-tagalog", "mbert", "xlmr"]
rows = []
for m in NAMES:
    log = json.load(open(R / m / "training_log.json"))
    ev = [(x["epoch"], x["eval_loss"]) for x in log if "eval_loss" in x]
    best = min(ev, key=lambda e: e[1])
    summ = next((x for x in log if "train_runtime" in x), {})
    rows.append({"model": m, "epochs run": int(ev[-1][0]), "best epoch": int(best[0]), "best validation loss": round(best[1], 4),
                 "training time (min)": round(summ.get("train_runtime", float("nan")) / 60, 1)})
display(pd.DataFrame(rows))

## 5. Internal test results
The internal test set has 3,803 held-out news articles. The first table gives each model's metrics (not credible = positive class). The second gives 95% bootstrap confidence intervals (2,000 resamples). The third is the **McNemar test**, which asks whether two models make significantly different errors on the same articles.

**Reading the tie.** RoBERTa-Tagalog and XLM-RoBERTa have overlapping intervals and McNemar p = 0.55, so they are statistically tied. mBERT is lower than both. A plain **TF-IDF + linear baseline** (no deep model) scores close to the transformers, which tells us the internal task is partly solvable with surface word patterns. All numbers: one seed, one run.

In [ ]:
models = NAMES + ["baseline-tfidf"]
cols = {"accuracy": "Acc", "precision_not_credible": "Precision (NC)", "recall_not_credible": "Recall (NC)",
        "f1_not_credible": "F1 (NC)", "macro_f1": "Macro F1", "weighted_f1": "Weighted F1"}
met = pd.DataFrame({m: {v: json.load(open(R / m / "metrics.json"))[k] for k, v in cols.items()} for m in models}).T.round(4)
display(Markdown("**Metrics on the internal test set** (`results/<model>/metrics.json`)")); display(met)

ci = json.load(open(R / "bootstrap_ci.json"))
cirows = [{"model": m, "accuracy": round(v["accuracy"], 4), "accuracy 95% CI": "[%.4f, %.4f]" % tuple(v["accuracy_ci95"]),
           "weighted F1": round(v["weighted_f1"], 4), "weighted F1 95% CI": "[%.4f, %.4f]" % tuple(v["weighted_f1_ci95"])}
          for m, v in ci["per_model"].items()]
display(Markdown("**95%% bootstrap CIs** (`results/bootstrap_ci.json`, n = %d, %d resamples)" % (ci["n_test"], ci["resamples"]))); display(pd.DataFrame(cirows))

txt = open(R / "comparison.md", encoding="utf8").read()
i = txt.find("**McNemar"); j = txt.find("## Supplementary")
display(Markdown("**McNemar** (`results/comparison.md`)\n\n" + txt[i:j]))
for f in ["fig_a_internal_metrics.png", "fig_b_confusion_matrices.png"]:
    if (R / "figures" / f).exists(): display(Image(filename=str(R / "figures" / f)))

## 6. Limits of the internal scores
High internal scores do not mean the models understand misinformation. These checks show why. One plain sentence per figure:
- **Leakage probe (fig c).** A simple classifier that sees only the first 12 or last 12 words still scores far above guessing, so short fragments already carry source-style cues (a way the outlet writes), not just meaning.
- **Overconfidence (fig d).** When the models are wrong they are often still very sure (see the median confidence when wrong below), so confidence is not a calibrated probability.
- **Length falloff (fig e / `domain_shift.json`).** Accuracy changes when test articles are cut to their first 15, 30 or 60 words; shown only once that Colab run has been done.
- **Length distribution (fig f).** Training articles are long; collected social posts are much shorter, so the models face a domain they were not trained on.

In [ ]:
lp = json.load(open(R / "leakage_probe.json"))
display(pd.DataFrame({"accuracy": {"always predict the majority class": lp["majority_class_accuracy"],
    "TF-IDF, full text": lp["full_text"]["accuracy"], "TF-IDF, first 12 words only": lp["first_12_words"]["accuracy"],
    "TF-IDF, last 12 words only": lp["last_12_words"]["accuracy"]}}).round(4))
cc = json.load(open(R / "confidence_check.json"))["models"]
display(pd.DataFrame(cc).T[["errors", "median_confidence_when_wrong", "median_confidence_when_right",
                            "errors_with_confidence_at_least_0.99"]])
if (R / "domain_shift.json").exists():
    display(Markdown("**domain_shift.json**")); print(json.dumps(json.load(open(R / "domain_shift.json")), indent=1)[:2500])
else:
    print("Length falloff (domain_shift.json): pending, needs the Colab run in colab_eval.ipynb section 2b.")
for f in ["fig_c_leakage_probe.png", "fig_d_confidence_internal.png", "fig_e_accuracy_vs_words_kept.png", "fig_f_length_distribution.png"]:
    if (R / "figures" / f).exists(): display(Image(filename=str(R / "figures" / f)))

## 7. Taglish results
The Taglish election posts are the real test of how the models cope with short social posts. They are **test only**. Results appear below once the group's annotation is finished and `colab_eval.ipynb` has been run; until then the cell prints "pending: annotation".

In [ ]:
ext = sorted(R.glob("external_*")) + sorted(R.glob("*/external_*"))
tags = [p for p in (R / "figures").glob("fig_[g-j]_*.png")]
if not ext and not tags:
    print("pending: annotation (no Taglish result files in results/ yet)")
else:
    for p in ext:
        if p.suffix == ".md": display(Markdown("**" + str(p) + "**\n\n" + p.read_text(encoding="utf8")))
        elif p.suffix == ".json" and p.stat().st_size < 6000: print("---", p); print(p.read_text(encoding="utf8")[:2500])
    for p in sorted(tags): display(Image(filename=str(p)))

## 8. Live prediction
All three models score the same post. Models are downloaded from Hugging Face on first use (about 1 to 2 GB, a minute or two) and then stay in memory. `P(credible)` and `P(not credible)` add up to 1. The first five posts are **real public posts** from news outlets and an official account (2022), copied with their links from the collected candidate set; no private users. We do not show a "true" label for them: Taglish labels are not final.

Reading rule: a verdict says how a post resembles what the source labels looked like, not whether it is true.

**Expect most short posts to come out as "not credible", even ones from news outlets.** The models were trained on long articles, so short posts are outside what they know (sections 5 and 6). That is the domain-shift limit, not evidence that these posts are false.

In [ ]:
import sys
sys.path.insert(0, "src")
import predict   # loads configs/config.yaml from the repo root
LOADED = {m: predict.load(m, "hub") for m in NAMES}

POSTS = [
    ("News report, 10 words (ABS-CBN News)", "https://x.com/ABSCBNNews/status/1510848386475311105",
     "Comelec eyes debate skipping as basis for candidate disqualification #Halalan2022"),
    ("News report, Filipino, 16 words (dzBB)", "https://x.com/dzbb/status/1523091909262004224",
     "#Eleksyon2022: Ilang mga nagsimba sa Pasig Cathedral ngayong Linggo, ipinanalangin ang payapang eleksyon. | via \n@MMakalalad"),
    ("Official account notice, 23 words (COMELEC)", "https://twitter.com/COMELEC/status/1522093742202707968",
     "Fully shade the oval to the LEFT of your candidate's name.\n\nDo NOT make any other mark on the ballot.#VoteSAFEPilipinas #BumotoKa #NLE2022 pic.twitter.com/g2t1EJLcTP"),
    ("News report, Taglish, 46 words (ABS-CBN News)", "https://x.com/ABSCBNNews/status/1523447780689080320",
     "NOW: Comelec holds press briefing over an hour after polls opened.\n\n\u201cAs early as 6 am marami na tayong kababayang nagpunta sa polling places\u2026 Tuloy ang botohan ano man ang mangyari, we have plans B, C, & other contingency plans,\u201d says acting spox John Laudiangco. #Halalan2022"),
    ("News report, list, 48 words (GMA News)", "https://x.com/gmanews/status/1523494357898493953",
     "JUST IN: Comelec Commissioner George Garcia says more than 1,800 VCMs have malfunctioned. \n\nCommon issues [but resolved]\n- Paper Jam - 940\n- Rejected ballots - 606\n- VCM scanner - 158\n- VCM printer not printing - 87\n- Not printing properly -76 #Eleksyon2022 | via \n@tinapperez"),
]

def score(text):
    rows = []
    for m, (tok, model) in LOADED.items():
        p = predict.predict(tok, model, [text])[0]      # [P(not credible), P(credible)]
        rows.append({"model": m, "verdict": "credible" if p[1] >= 0.5 else "NOT credible",
                     "P(credible)": round(float(p[1]), 3), "P(not credible)": round(float(p[0]), 3)})
    return pd.DataFrame(rows)

for name, url, text in POSTS:
    display(Markdown("**%s**  \nSource: %s  \n> %s" % (name, url, text.replace("\n", " "))))
    display(score(text))

**Try a post the panel suggests.** Paste it into the form field in the next cell (right side of the cell) and run it. Use only real public text; do not paste private messages.

In [ ]:
custom_text = "Totoo ba na may dayaan sa eleksyon? Share this para malaman ng lahat!" #@param {type:"string"}
display(Markdown("> " + custom_text)); display(score(custom_text))

## 9. Integrated Gradients for one post
Integrated Gradients (IG) compares the post with a blank version of itself (every word replaced by padding) and adds up how much each word pushed the model toward **not credible** (positive) or **credible** (negative). So IG shows **what a model relied on**, not what is true, and it can change with another training seed. Scores are summed per whitespace word so the three tokenizers can be compared. IG starts at 100 steps and doubles until its own completeness check is within 5% (capped here at 400 to keep it quick; on CPU this cell can take a few minutes).

In [ ]:
import interpret
IG_POST = POSTS[3][2]
display(Markdown("> " + IG_POST.replace("\n", " ")))
for m, (tok, model) in LOADED.items():
    r = interpret.explain(tok, model, IG_POST, n_steps=100, max_steps=400)
    ws = sorted(r["words"], key=lambda w: w["attr"])
    c = r["completeness"]
    display(Markdown("**%s**: %s (P(not credible) = %.3f). IG steps %d, completeness error %.1f%%" % (m, r["pred_name"], r["prob_not_credible"], c["n_steps"], 100 * c["rel_error"])))
    display(pd.DataFrame({"toward NOT credible": [f'{w["word"]} ({w["attr"]:+.2f})' for w in ws[::-1][:6]],
                          "toward credible": [f'{w["word"]} ({w["attr"]:+.2f})' for w in ws[:6]]}))

## 10. Summary
Key findings with the file each number comes from. Numbers are printed from the files, not typed in.

In [ ]:
import re
bm = ci["per_model"]
m_rx = re.search(r"\| roberta-tagalog vs xlmr \| (\d+) \| (\d+) \| ([0-9.e-]+) \|", txt)
lines = [
    ("Fine-tuning", "One fixed configuration, no hyperparameter search, single seed (seed %d, lr %s, max %d epochs, early stopping)" % (cfg["seed"], t["learning_rate"], t["max_epochs"]), "configs/config.yaml"),
    ("Internal accuracy", ", ".join("%s %.3f" % (m, bm[m]["accuracy"]) for m in bm), "results/bootstrap_ci.json"),
    ("RoBERTa-Tagalog vs XLM-R", "McNemar p = %s (b = %s, c = %s): not significant, not ranked" % (m_rx.group(3), m_rx.group(1), m_rx.group(2)), "results/comparison.md"),
    ("Simple baseline", "TF-IDF baseline accuracy %.3f, close to the transformers" % bm["baseline-tfidf"]["accuracy"], "results/bootstrap_ci.json"),
    ("Leakage probe", "First 12 words alone: accuracy %.3f vs majority-class %.3f" % (lp["first_12_words"]["accuracy"], lp["majority_class_accuracy"]), "results/leakage_probe.json"),
    ("Overconfidence", "Errors at 0.99+ confidence: " + ", ".join("%s %d" % (m, v["errors_with_confidence_at_least_0.99"]) for m, v in cc.items()), "results/confidence_check.json"),
    ("Taglish posts", "pending: annotation" if not ext else "see section 7", "results/external_*"),
]
display(pd.DataFrame(lines, columns=["Finding", "What the file says", "Source file"]))
display(Markdown("**Limits.** Labels follow the source, so these scores are not proof of detecting misinformation. One seed per model. IG is descriptive, not causal."))